# Unit 4, Lecture 4: Shared state and the blackboard

So far, agents passed a single **message** down an edge: one hands the next a
result, and it is gone once read. Real collaboration is richer. A summary agent
needs to see what triage **and** security **and** priority all found, not just
the one message that reached it.

The pattern is a **blackboard**: a shared space every agent reads from and writes
to. One agent posts a finding, another reads it and adds its own, a final agent
reads the whole board.

Built on the real `ctx.set_state` / `ctx.get_state`, running **offline**, because
the agents are plain functions writing to shared state.

## The board: post and read

`set_state(key, value)` posts to the shared board; `get_state(key)` reads it. Two
calls are the whole API. Every agent in the workflow shares the same board.

In [ ]:
from cse476.blackboard import build_blackboard, TICKET, FINDINGS

wf = build_blackboard()
print("blackboard workflow built:", type(wf).__name__)
print()
print("the board keys every agent agrees on:")
print("  TICKET   =", repr(TICKET))
print("  FINDINGS =", repr(FINDINGS))

## Read, modify, write

The core cycle. Each scanning agent **reads** the current findings, **adds** its
own, and **writes** the list back. The critical habit: append, do not overwrite.
Skip the read and you erase everyone else's work.

In [ ]:
import inspect
from cse476.blackboard import security_scan

# see the read-modify-write cycle in one agent
print(inspect.getsource(security_scan))

## Run it: the board fills up

In [ ]:
from cse476.blackboard import run_blackboard

print(await run_blackboard("someone tried to hack my account, this is urgent"))
print()
print(await run_blackboard("just a small question about my invoice"))

The final `summarise` agent read **nothing** from its incoming message.
Everything in that board, the ticket, the security finding, the priority finding,
came off the shared state, written by the agents before it. That is the
blackboard: the edges carry control, the state carries the collaboration.

## Watch findings accumulate, in order

Each agent builds on the ones before it. Because `priority_scan` runs after
`security_scan` on the same board, by the time priority reads the findings, the
security verdict is already there.

In [ ]:
out = await run_blackboard("hacked and urgent")
board = out.split("Board:")[1].strip()
print("final board:", board)
print()
print("security appears before priority:",
      board.index("security") < board.index("priority"))
print("both survived (append, not overwrite): security AND priority both present")

## The overwrite bug (cause it on purpose)

The most common blackboard mistake: an agent that **skips the read** and just
writes its own finding, erasing everyone else's. Build one and watch the earlier
findings vanish.

In [ ]:
from agent_framework import WorkflowBuilder, WorkflowContext, executor
from cse476.blackboard import intake, security_scan, FINDINGS

# a BROKEN agent: it overwrites instead of appending
@executor(id="clobber")
async def clobber(ticket: str, ctx: WorkflowContext[str]) -> None:
    # BUG: writes only its own finding, without reading what is there first
    ctx.set_state(FINDINGS, ["priority: HIGH"])   # erases security's finding!
    await ctx.send_message(ticket)

@executor(id="show")
async def show(ticket: str, ctx: WorkflowContext) -> None:
    await ctx.yield_output(f"board: {ctx.get_state(FINDINGS)}")

broken = (WorkflowBuilder(start_executor=intake)
    .add_edge(intake, security_scan)
    .add_edge(security_scan, clobber)   # this clobbers security's finding
    .add_edge(clobber, show)
    .build())

result = await broken.run("someone hacked my account")
print(result.get_outputs()[0])
print()
print("Security's 'RISK' finding is GONE. clobber overwrote instead of appending.")
print("This is why you always READ before you WRITE.")

## The mapping, and message vs board

In [ ]:
from cse476.blackboard import BLACKBOARD_MAP, message_vs_blackboard

for concept, tie in BLACKBOARD_MAP.items():
    print(f"{concept:28} ->  {tie}")
print()
for k, v in message_vs_blackboard().items():
    print(f"{k:14}: {v}")

A board is not strictly better than a message; it is more powerful and needs
more care. Use a **message** when one agent hands one result to the next. Use a
**board** when several agents build on a shared, growing picture. The cost of a
board is discipline: agreed key names, and append instead of overwrite.

## Your turn

**1. Add a board writer.** Add a sentiment scan between priority and summary.
Have it read the board, append its finding, write back. Confirm the summary now
shows three findings, all preserved.

**2. Cause the bug on purpose.** Write an agent that skips the read and calls
`set_state` with only its own finding. Watch earlier findings vanish. Then fix it
to read first. You have now seen the overwrite bug.

**3. Message or board?** In your capstone, name one place a plain message is right
and one place a shared board is right. One sentence each on why.

In [ ]:
# your work here
